# V0.9 Persistent Memory Lab

问题：一个 Agent 在 Session A 里记住的长期语义信息，为什么能在 Session B 里继续使用？

本 lab 逐格展示：Memory survives Session boundaries；Memory != Session；Memory != Context；Memory ID != authority；Forget 不会重写历史 Session。

术语：

- Session：一次任务/交互的 durable event journal。
- Memory：跨 Session 的长期语义记忆。
- Context：模型本轮可见的有界投影。
- Provenance：这条 Memory 从哪里来。
- Capability：当前读取/写入/遗忘 Memory 的授权。

In [ ]:
MODE = "deterministic"  # deterministic 是可复现实验；real_model 可由上层演示只负责 proposal/query。

from pathlib import Path
from tempfile import TemporaryDirectory

from agentkernel import (
    AuthorizationRequest,
    CapabilityEvaluator,
    CapabilityGrant,
    JsonlMemoryStore,
    MEMORY_FORGET_ACTION,
    MEMORY_READ_ACTION,
    MEMORY_WRITE_ACTION,
    MemoryAccessDenied,
    MemoryProvenance,
    MemoryService,
    memory_namespace_scope,
    project_memories_to_context_pages,
)

AGENT_A = "lab-agent-a"
AGENT_B = "lab-agent-b"
NAMESPACE = "preferences"
SESSION_A = "session-A"
SESSION_B = "session-B"

tmp = TemporaryDirectory()
store_path = Path(tmp.name) / "memory.jsonl"

def grants(agent_id, *actions):
    return CapabilityEvaluator(
        CapabilityGrant(agent_id, action, memory_namespace_scope(AGENT_A, NAMESPACE))
        for action in actions
    )

def show(title, payload):
    print(f"\n=== {title} ===")
    if isinstance(payload, dict):
        for key, value in payload.items():
            print(f"{key}: {value}")
    else:
        print(payload)

show("Setup", {"mode": MODE, "store_path": store_path, "session_a": SESSION_A, "session_b": SESSION_B})

## Step 1: Session A 产生显式记忆 proposal

这里用 deterministic proposal 代替真实模型输出。真实模型模式也只能提出 proposal，不能绕过 Kernel 写 MemoryStore。

In [ ]:
user_message = "以后代码示例优先使用 Python。"
memory_proposal = {
    "namespace": NAMESPACE,
    "content": "用户偏好使用 Python 编写代码示例。",
    "source_session_id": SESSION_A,
    "source_event_id": "event-user-1",
}
show("Session A proposal", {"user": user_message, "proposal": memory_proposal})

## Step 2: Kernel 授权

`memory.write` 是 Capability，不是 prompt 文本。模型不能给自己授权。

In [ ]:
write_auth = grants(AGENT_A, MEMORY_WRITE_ACTION)
decision = write_auth.authorize(
    AuthorizationRequest(
        agent_id=AGENT_A,
        action=MEMORY_WRITE_ACTION,
        resource=memory_namespace_scope(AGENT_A, NAMESPACE),
    )
)
show("Authorization", {"agent_id": AGENT_A, "action": MEMORY_WRITE_ACTION, "allowed": decision.allowed, "reason": decision.reason})

## Step 3: Durable Memory write

写入的是 append-only MemoryEvent。`MemoryRecord` 的存在只说明 Kernel 记住了这个 proposition，不说明 proposition 是客观真理。

In [ ]:
memory_a = MemoryService(JsonlMemoryStore(store_path))
record = memory_a.remember(
    agent_id=AGENT_A,
    namespace=NAMESPACE,
    content=memory_proposal["content"],
    provenance=MemoryProvenance(
        source="session",
        source_session_id=SESSION_A,
        source_event_id=memory_proposal["source_event_id"],
        source_agent_id=AGENT_A,
    ),
    capability_evaluator=write_auth,
)
show("Memory written", {"memory_id": record.memory_id, "uri": record.uri, "source_session_id": record.provenance.source_session_id})

## Step 4: 关闭 Session A / Runtime A

现在模拟任务结束。MemoryStore 的 durable event 留在 JSONL 中。

In [ ]:
memory_a.close()
show("Runtime A closed", {"session_a": SESSION_A, "memory_store_exists": store_path.exists(), "store_bytes": store_path.stat().st_size})

## Step 5: Fresh Session B / Runtime B

新的 runtime、新的 Session，但使用同一个 durable MemoryStore。

In [ ]:
memory_b = MemoryService(JsonlMemoryStore(store_path))
show("Runtime B", {"session_a_id": SESSION_A, "session_b_id": SESSION_B, "same_session": SESSION_A == SESSION_B, "same_memory_id_expected": record.memory_id})

## Step 6: Retrieval

Memory retrieval 需要当前 `memory.read` capability。

In [ ]:
results = memory_b.search(
    agent_id=AGENT_A,
    owner_agent_id=AGENT_A,
    namespace=NAMESPACE,
    query="Python 代码示例",
    limit=5,
    capability_evaluator=grants(AGENT_A, MEMORY_READ_ACTION),
)
show("Search result", {"count": len(results), "memory_ids": [item.memory_id for item in results], "contents": [item.content for item in results]})

## Step 7: Context projection

Memory 不会自动全部塞进 Context。这里只投影选中的 top-k records。

In [ ]:
projection = project_memories_to_context_pages(results, top_k=1, total_memory_records=1)
show("Context projection", {"selected_count": projection.selected_count, "page_count": len(projection.pages), "model_visible_page": projection.pages[0].content})

## Step 8: Memory ID != Permission

Agent B 即使知道 `memory_id`，没有 current capability 也不能读取。

In [ ]:
denied = False
try:
    memory_b.read(record.memory_id, agent_id=AGENT_B, capability_evaluator=CapabilityEvaluator(()))
except MemoryAccessDenied as error:
    denied = True
    denial_reason = str(error)
show("Unauthorized read", {"agent_b_knows_memory_id": record.memory_id, "denied": denied, "reason": denial_reason})

## Step 9: Forget

Forget 是 semantic inactive，不是物理擦除，也不重写历史 Session。

In [ ]:
memory_b.forget(record.uri, agent_id=AGENT_A, capability_evaluator=grants(AGENT_A, MEMORY_FORGET_ACTION))
memory_b.close()

memory_c = MemoryService(JsonlMemoryStore(store_path))
active_after_forget = memory_c.search(
    agent_id=AGENT_A,
    owner_agent_id=AGENT_A,
    namespace=NAMESPACE,
    query="Python",
    limit=5,
    capability_evaluator=grants(AGENT_A, MEMORY_READ_ACTION),
)
inactive = memory_c.read(record.uri, agent_id=AGENT_A, include_inactive=True, capability_evaluator=grants(AGENT_A, MEMORY_READ_ACTION))
show("After forget + restart", {"active_search_count": len(active_after_forget), "inactive_record_exists": inactive.memory_id, "active_flag": inactive.active, "forgotten_at": inactive.forgotten_at})

## Final

这个实验证明：

- Memory survives Session boundaries.
- Memory != Session.
- Memory != Context.
- Memory ID != authority.
- Forget changes active memory state without rewriting historical Session.

这个实验不证明：

- Memory content objectively true.
- Secure physical deletion.
- Vector/RAG retrieval quality.
- Distributed persistence correctness.